# Setup

In [1]:
import numpy as np
from pathlib import Path
import pandas as pd
import torch
import random
from utils.utils import *
from data.simulate_walk_the_book import *
from utils.datastuff import TrainCfg
from utils.train import train_val
from utils.test import generate_test_loader, generate_test_predictions
from data.simulate_walk_the_book import simulate_walk_the_book
import warnings


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

# Fix randomness for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

warnings.filterwarnings("ignore", category=UserWarning)

device: cpu


In [2]:

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

In [3]:
# are we consuming asks or bids?
side = "ask" # if positions are positive, we consume asks, if negative we consume bids.
ASK_PRICE_COLS = ['ask_price_1', 'ask_price_2', 'ask_price_3', 'ask_price_4', 'ask_price_5']
ASK_VOL_COLS = ['ask_vol_1', 'ask_vol_2', 'ask_vol_3', 'ask_vol_4', 'ask_vol_5']
BID_PRICE_COLS = ['bid_price_1', 'bid_price_2', 'bid_price_3', 'bid_price_4', 'bid_price_5']
BID_VOL_COLS = ['bid_vol_1', 'bid_vol_2', 'bid_vol_3', 'bid_vol_4', 'bid_vol_5']

In [4]:
import sys, os

# Paths and volume_to_fill
# root_path = DATA_PATH
root_path = "../data"
root = Path(root_path) if Path(root_path).exists() else Path.cwd()

import sys
if str(root / "src") not in sys.path:
    sys.path.append(str(root / "src"))

data_assets = [folder_name for folder_name in os.listdir(root) if "USDT" in folder_name]

X_dfs = {}
y_dfs = {}
assets_ids = {}

for data_asset in data_assets:

    symbol_dir = root / data_asset
    X_path = symbol_dir / "X_train.parquet"
    Y_path = symbol_dir / "y_train.parquet"
    X_test_path = symbol_dir / "X_test.parquet"
    vol_path = symbol_dir / "vol_to_fill.txt"

    volume_to_fill = None
    if vol_path.exists():
        import re
        with open(vol_path) as f:
            m = re.search(r"([\d.]+)", f.read())
        if m:
            volume_to_fill = float(m.group(1))

    X_df = pd.read_parquet(X_path)
    y_df = pd.read_parquet(Y_path)

    ids = pd.Series(X_df["anonymized_id"].unique(), name="anonymized_id")

    last_time = X_df["time_in_hour"].sort_values().iloc[-1]
    future_times = pd.Series(
        [last_time + pd.Timedelta(seconds=i) for i in range(1, 61)],
        name="time_in_hour"
    )

    past_times = pd.Series(
        X_df["time_in_hour"].sort_values().unique(),
        name="time_in_hour"
    )

    full_grid = (
        ids.to_frame()
        .merge(future_times.to_frame(), how="cross")
    )

    full_grid_X = (
        ids.to_frame()
        .merge(past_times.to_frame(), how="cross")
    )

    y_df = full_grid.merge(
        y_df,
        on=["anonymized_id", "time_in_hour"],
        how="left"
    )

    X_df = full_grid_X.merge(
        X_df,
        on=["anonymized_id", "time_in_hour"],
        how="left"
    )

    # price imputation
    price_cols = ASK_PRICE_COLS + BID_PRICE_COLS + ["close"]
    X_df[price_cols] = X_df.groupby("anonymized_id")[price_cols].transform(lambda x: x.ffill().bfill())
    y_df[price_cols] = y_df.groupby("anonymized_id")[price_cols].transform(lambda x: x.ffill().bfill())

    # volume imputation
    volume_cols = BID_VOL_COLS + ASK_VOL_COLS + ["volume"]
    X_df[volume_cols] = X_df[volume_cols].fillna(0)
    y_df[volume_cols] = y_df[volume_cols].fillna(0)

    X_dfs[data_asset] = X_df
    y_dfs[data_asset] = y_df
    assets_ids[data_asset] = ids

In [5]:
volumes_to_fill = {}

for asset in data_assets:
    vol_path = root / asset / "vol_to_fill.txt"
    if vol_path.exists():
        with open(vol_path) as f:
            m = re.search(r"([\d.]+)", f.read())
        if m:            
            volumes_to_fill[asset] = float(m.group(1)) 

import matplotlib.pyplot as plt

def plot_volumes_to_fill():
    # Sort dictionary by values
    sorted_items = sorted(volumes_to_fill.items(), key=lambda x: x[1])

    # Unzip keys and values
    labels, values = zip(*sorted_items)

    # Create bar plot
    plt.figure()
    plt.bar(labels, values)

    # Optional labels
    plt.xlabel("Items")
    plt.ylabel("Values")
    plt.title("Volumes to Fill by Asset")

    plt.show()

In [6]:
df_k = pd.DataFrame({
    "Asset": [
        "BTCUSDT", "ETHUSDT", "LTCUSDT",
        "SOLUSDT", "ADAUSDT", "DOGEUSDT", "XRPUSDT"
    ],
    "Optimal_K": [
        14, 26, 16,
        17, 7, 20, 20
    ],
    "bps_optimal_K": [
        1.323212, 2.712073, 4.819771,
        5.343504, 4.628115, 4.976437, 3.567023
    ]
})

df_k

,Asset,Optimal_K,bps_optimal_K
0,BTCUSDT,14,1.323212
1,ETHUSDT,26,2.712073
2,LTCUSDT,16,4.819771
3,SOLUSDT,17,5.343504
4,ADAUSDT,7,4.628115
5,DOGEUSDT,20,4.976437
6,XRPUSDT,20,3.567023


In [7]:
optimal_k = df_k[df_k["Asset"] == data_asset]["Optimal_K"].iloc[0]

# Ultimate Benchmark of Above Approaches

we want a MultiIndex Asset - Model \in {optimal K, capping(funcs), redistribute(funcs), redistribute(decreasing caps), redistribute(decreasing caps with slope)}

features are [bps, % filled]

In [8]:
benchmark = pd.DataFrame(
    columns=["bps", "percent_filled"]
)

benchmark.index = pd.MultiIndex.from_tuples(
    [],
    names=["Asset", "Volume to Fill", "strategy"]
)

benchmark

,,,bps,percent_filled
Asset,Volume to Fill,strategy,,


In [9]:
for data_asset in data_assets:

    X_df = X_dfs[data_asset]
    y_df = y_dfs[data_asset]
    ids = assets_ids[data_asset]
    optimal_k = df_k[df_k["Asset"] == data_asset]["Optimal_K"].iloc[0]
    volume_to_fill = volumes_to_fill[data_asset]

    bpss = []
    percents_filled = []

    for anon_id in ids:

        df_inst = y_df[y_df["anonymized_id"] == anon_id].sort_values("time_in_hour")

        ask_prices = df_inst[ASK_PRICE_COLS].to_numpy()
        ask_vols = df_inst[ASK_VOL_COLS].to_numpy()
        bid_prices = df_inst[BID_PRICE_COLS].to_numpy()
        bid_vols = df_inst[BID_VOL_COLS].to_numpy()
        close_price = df_inst['close'].dropna().iloc[-1]

        weights = np.array([0.]* (60-optimal_k) + [1.]* optimal_k)
        weights /= weights.sum()

        if not np.isclose(weights.sum(), 1):
            print(f"{anon_id} doesn't have weights that sum to 1.")

        id_positions = weights * volume_to_fill

        model_vol, model_avg_price = simulate_walk_the_book(
            id_positions, ask_prices, ask_vols, bid_prices, bid_vols
        )
        
        if model_vol > 0 and not np.isnan(model_avg_price):
            rel_error = np.abs(model_avg_price - close_price) / close_price
            vol_penalty = min(100.0, volume_to_fill / model_vol)
            bpss.append(rel_error * vol_penalty * 10000)
            percents_filled.append(model_vol / volume_to_fill * 100)

    strategy = "optimal_k"
    benchmark.loc[(data_asset, volume_to_fill, strategy), "bps"] = np.mean(bpss)
    benchmark.loc[(data_asset, volume_to_fill, strategy), "percent_filled"] = np.round(np.mean(percents_filled), 1)


In [10]:
benchmark

,,,bps,percent_filled
Asset,Volume to Fill,strategy,,
BTCUSDT,4.0,optimal_k,1.218637,99.7
XRPUSDT,13736.0,optimal_k,3.625474,100.0
ETHUSDT,55.0,optimal_k,2.66769,99.9
ADAUSDT,10436.0,optimal_k,5.686933,98.4
SOLUSDT,315.0,optimal_k,5.289195,100.0
DOGEUSDT,60349.0,optimal_k,5.000712,99.9
LTCUSDT,51.0,optimal_k,5.302726,99.4


## Cap the Allocated Volume Based on Previous Volume

capping analysis:
- plotting function for objective components
- plotting pipeline for redistribution & capping with linear, **2, **3, **4, **5

In [11]:
methods = ["capped", "distributed"]
functions = ["linear", "**2", "**3", "**5", "**10"]
percentiles = ["mean", 50, 80]


In [12]:
cap_col = f"{side}_vol_1"

for data_asset in data_assets:

    X_df = X_dfs[data_asset]
    y_df = y_dfs[data_asset]
    ids = assets_ids[data_asset]
    volume_to_fill = volumes_to_fill[data_asset]

    for percentile in percentiles:
        for func in functions:

            bpss = []
            percents_filled = []

            for anon_id in ids:

                df_inst = y_df[y_df["anonymized_id"] == anon_id].sort_values("time_in_hour")
                X_df_inst = X_df[X_df["anonymized_id"] == anon_id].sort_values("time_in_hour")

                if percentile == "mean":
                    volume_cap = np.mean(X_df_inst[cap_col])
                else:
                    volume_cap = np.percentile(X_df_inst[cap_col], percentile)

                ask_prices = df_inst[ASK_PRICE_COLS].to_numpy()
                ask_vols = df_inst[ASK_VOL_COLS].to_numpy()
                bid_prices = df_inst[BID_PRICE_COLS].to_numpy()
                bid_vols = df_inst[BID_VOL_COLS].to_numpy()
                close_price = df_inst['close'].dropna().iloc[-1]
                
                if volume_cap == 0:
                    print("mean used!!!")
                    volume_cap = np.mean(X_df_inst[cap_col])

                weights = np.linspace(0, 1, 60)

                if func != "linear":
                    weights = weights ** int(func[2:])

                weights /= weights.sum()

                if not np.isclose(weights.sum(), 1):
                    print(f"{anon_id} doesn't have weights that sum to 1.")

                # simply cap
                id_positions = np.minimum(weights * volume_to_fill, volume_cap)

                model_vol, model_avg_price = simulate_walk_the_book(
                    id_positions, ask_prices, ask_vols, bid_prices, bid_vols
                )
                
                if model_vol > 0 and not np.isnan(model_avg_price):
                    rel_error = np.abs(model_avg_price - close_price) / close_price
                    vol_penalty = min(100.0, volume_to_fill / model_vol)

                    bpss.append(rel_error * vol_penalty * 10000)
                    percents_filled.append(model_vol / volume_to_fill * 100)

            strategy = f"capped_{func}_{percentile}"
            benchmark.loc[(data_asset, volume_to_fill, strategy), "bps"] = np.mean(bpss)
            benchmark.loc[(data_asset, volume_to_fill, strategy), "percent_filled"] = np.round(np.mean(percents_filled), 1)



/var/folders/gh/k7rgm7qs3zl2lw8bn_ydlg_40000gq/T/ipykernel_2231/993677803.py:62: PerformanceWarning: indexing past lexsort depth may impact performance.
  benchmark.loc[(data_asset, volume_to_fill, strategy), "percent_filled"] = np.round(np.mean(percents_filled), 1)
/var/folders/gh/k7rgm7qs3zl2lw8bn_ydlg_40000gq/T/ipykernel_2231/993677803.py:61: PerformanceWarning: indexing past lexsort depth may impact performance.
  benchmark.loc[(data_asset, volume_to_fill, strategy), "bps"] = np.mean(bpss)
/var/folders/gh/k7rgm7qs3zl2lw8bn_ydlg_40000gq/T/ipykernel_2231/993677803.py:62: PerformanceWarning: indexing past lexsort depth may impact performance.
  benchmark.loc[(data_asset, volume_to_fill, strategy), "percent_filled"] = np.round(np.mean(percents_filled), 1)
/var/folders/gh/k7rgm7qs3zl2lw8bn_ydlg_40000gq/T/ipykernel_2231/993677803.py:61: PerformanceWarning: indexing past lexsort depth may impact performance.
  benchmark.loc[(data_asset, volume_to_fill, strategy), "bps"] = np.mean(bpss)
/v

## Redistribute if Cap Exceeded

**cap**(**function**) + **redistribute**()

In [13]:
def capped_allocation(weights, total_volume, cap):
    weights = np.asarray(weights, dtype=float)
    weights = weights / weights.sum()

    allocation = weights * total_volume
    capped = np.minimum(allocation, cap)

    deficit = total_volume - capped.sum()

    while deficit > 1e-9:
        free = capped < cap
        if not np.any(free):
            break

        w_free = weights[free]
        w_free = w_free / w_free.sum()

        increment = np.zeros_like(capped)
        increment[free] = w_free * deficit

        new_vals = np.minimum(capped + increment, cap)
        delta = new_vals - capped

        capped = new_vals
        deficit -= delta.sum()

    return capped


cap_col = f"{side}_vol_1"

for data_asset in data_assets:

    X_df = X_dfs[data_asset]
    y_df = y_dfs[data_asset]
    ids = assets_ids[data_asset]
    volume_to_fill = volumes_to_fill[data_asset]

    for percentile in percentiles:
        for func in functions:

            bpss = []
            percents_filled = []

            for anon_id in ids:

                df_inst = y_df[y_df["anonymized_id"] == anon_id].sort_values("time_in_hour")
                X_df_inst = X_df[X_df["anonymized_id"] == anon_id].sort_values("time_in_hour")

                if percentile == "mean":
                    volume_cap = np.mean(X_df_inst[cap_col])
                else:
                    volume_cap = np.percentile(X_df_inst[cap_col], percentile)

                ask_prices = df_inst[ASK_PRICE_COLS].to_numpy()
                ask_vols = df_inst[ASK_VOL_COLS].to_numpy()
                bid_prices = df_inst[BID_PRICE_COLS].to_numpy()
                bid_vols = df_inst[BID_VOL_COLS].to_numpy()
                close_price = df_inst['close'].dropna().iloc[-1]
                
                if volume_cap == 0:
                    print("mean used!!!")
                    volume_cap = np.mean(X_df_inst[cap_col])

                weights = np.linspace(0, 1, 60)

                if func != "linear":
                    weights = weights ** int(func[2:])

                id_positions = capped_allocation(weights, volume_to_fill, volume_cap)

                model_vol, model_avg_price = simulate_walk_the_book(
                    id_positions, ask_prices, ask_vols, bid_prices, bid_vols
                )
                
                if model_vol > 0 and not np.isnan(model_avg_price):
                    rel_error = np.abs(model_avg_price - close_price) / close_price
                    vol_penalty = min(100.0, volume_to_fill / model_vol)

                    bpss.append(rel_error * vol_penalty * 10000)
                    percents_filled.append(model_vol / volume_to_fill * 100)

            strategy = f"redistributed_{func}_{percentile}"
            benchmark.loc[(data_asset, volume_to_fill, strategy), "bps"] = np.mean(bpss)
            benchmark.loc[(data_asset, volume_to_fill, strategy), "percent_filled"] = np.round(np.mean(percents_filled), 1)

/var/folders/gh/k7rgm7qs3zl2lw8bn_ydlg_40000gq/T/ipykernel_2231/2444127509.py:84: PerformanceWarning: indexing past lexsort depth may impact performance.
  benchmark.loc[(data_asset, volume_to_fill, strategy), "bps"] = np.mean(bpss)
/var/folders/gh/k7rgm7qs3zl2lw8bn_ydlg_40000gq/T/ipykernel_2231/2444127509.py:85: PerformanceWarning: indexing past lexsort depth may impact performance.
  benchmark.loc[(data_asset, volume_to_fill, strategy), "percent_filled"] = np.round(np.mean(percents_filled), 1)
/var/folders/gh/k7rgm7qs3zl2lw8bn_ydlg_40000gq/T/ipykernel_2231/2444127509.py:84: PerformanceWarning: indexing past lexsort depth may impact performance.
  benchmark.loc[(data_asset, volume_to_fill, strategy), "bps"] = np.mean(bpss)
/var/folders/gh/k7rgm7qs3zl2lw8bn_ydlg_40000gq/T/ipykernel_2231/2444127509.py:85: PerformanceWarning: indexing past lexsort depth may impact performance.
  benchmark.loc[(data_asset, volume_to_fill, strategy), "percent_filled"] = np.round(np.mean(percents_filled), 1

In [14]:
benchmark.loc[data_assets[2]]

bps percent_filled
Volume to Fill strategy                                          
55.0           optimal_k                   2.66769           99.9
               capped_linear_mean         2.690608           99.5
               capped_**2_mean             2.70431           97.1
               capped_**3_mean            2.833386           92.5
               capped_**5_mean            3.215286           81.7
               capped_**10_mean           4.226643           61.6
               capped_linear_50           2.803868           98.0
               capped_**2_50              2.919799           92.8
               capped_**3_50               3.14148           86.0
               capped_**5_50              3.659139           73.5
               capped_**10_50             4.887437           53.8
               capped_linear_80           2.662127           99.9
               capped_**2_80              2.599004           99.4
               capped_**3_80              2.632277           97.9
               capped_**5_80              2.846299           91.9
               capped_**10_80             3.581672           74.2
               redistributed_linear_mean  2.676201          100.0
               redistributed_**2_mean     2.630856           99.9
               redistributed_**3_mean      2.63874           99.9
               redistributed_**5_mean     2.677028           99.8
               redistributed_**10_mean    2.732175           99.8
               redistributed_linear_50    2.747656           99.7
               redistributed_**2_50       2.720529           99.6
               redistributed_**3_50       2.727296           99.6
               redistributed_**5_50       2.755497           99.6
               redistributed_**10_50       2.78928           99.6
               redistributed_linear_80    2.660714          100.0
               redistributed_**2_80         2.5848           99.9
               redistributed_**3_80       2.583682           99.8
               redistributed_**5_80       2.645805           99.7
               redistributed_**10_80      2.756247           99.5

In [15]:
benchmark_copy = benchmark.copy()

## Redistribute if Cap Exceeded but Decrease Caps based on Distance to Final Sec

**cap**(**function**) + **redistribute**(**decreasing caps**)

In [16]:
second_caps_start = [20, 30, 40]

In [17]:
for data_asset in data_assets:

    X_df = X_dfs[data_asset]
    y_df = y_dfs[data_asset]
    ids = assets_ids[data_asset]
    volume_to_fill = volumes_to_fill[data_asset]

    for percentile in percentiles:
        for func in functions:
            for second_cap in second_caps_start:

                bpss = []
                percents_filled = []

                for anon_id in ids:

                    df_inst = y_df[y_df["anonymized_id"] == anon_id].sort_values("time_in_hour")
                    X_df_inst = X_df[X_df["anonymized_id"] == anon_id].sort_values("time_in_hour")

                    if percentile == "mean":
                        volume_cap = np.mean(X_df_inst[cap_col])
                    else:
                        volume_cap = np.percentile(X_df_inst[cap_col], percentile)

                    if volume_cap == 0:
                        print("mean used!!!")
                        volume_cap = np.mean(X_df_inst[cap_col])
                    
                    volume_caps = np.append(np.array([0]*second_cap), np.linspace(0, volume_cap, 60-second_cap)) # linearly increasing caps from second_cap onwards

                    ask_prices = df_inst[ASK_PRICE_COLS].to_numpy()
                    ask_vols = df_inst[ASK_VOL_COLS].to_numpy()
                    bid_prices = df_inst[BID_PRICE_COLS].to_numpy()
                    bid_vols = df_inst[BID_VOL_COLS].to_numpy()
                    close_price = df_inst['close'].dropna().iloc[-1]

                    weights = np.linspace(0, 1, 60)

                    if func != "linear":
                        weights = weights ** int(func[2:])

                    id_positions = capped_allocation(weights, volume_to_fill, volume_caps)

                    model_vol, model_avg_price = simulate_walk_the_book(
                        id_positions, ask_prices, ask_vols, bid_prices, bid_vols
                    )
                    
                    if model_vol > 0 and not np.isnan(model_avg_price):
                        rel_error = np.abs(model_avg_price - close_price) / close_price
                        vol_penalty = min(100.0, volume_to_fill / model_vol)

                        bpss.append(rel_error * vol_penalty * 10000)
                        percents_filled.append(model_vol / volume_to_fill * 100)

                strategy = f"redistributed_second{second_cap}_{func}_{percentile}"
                benchmark.loc[(data_asset, volume_to_fill, strategy), "bps"] = np.mean(bpss)
                benchmark.loc[(data_asset, volume_to_fill, strategy), "percent_filled"] = np.round(np.mean(percents_filled), 1)

/var/folders/gh/k7rgm7qs3zl2lw8bn_ydlg_40000gq/T/ipykernel_2231/2411745762.py:56: PerformanceWarning: indexing past lexsort depth may impact performance.
  benchmark.loc[(data_asset, volume_to_fill, strategy), "bps"] = np.mean(bpss)
/var/folders/gh/k7rgm7qs3zl2lw8bn_ydlg_40000gq/T/ipykernel_2231/2411745762.py:57: PerformanceWarning: indexing past lexsort depth may impact performance.
  benchmark.loc[(data_asset, volume_to_fill, strategy), "percent_filled"] = np.round(np.mean(percents_filled), 1)
/var/folders/gh/k7rgm7qs3zl2lw8bn_ydlg_40000gq/T/ipykernel_2231/2411745762.py:56: PerformanceWarning: indexing past lexsort depth may impact performance.
  benchmark.loc[(data_asset, volume_to_fill, strategy), "bps"] = np.mean(bpss)
/var/folders/gh/k7rgm7qs3zl2lw8bn_ydlg_40000gq/T/ipykernel_2231/2411745762.py:57: PerformanceWarning: indexing past lexsort depth may impact performance.
  benchmark.loc[(data_asset, volume_to_fill, strategy), "percent_filled"] = np.round(np.mean(percents_filled), 1

In [18]:
benchmark

bps percent_filled
Asset   Volume to Fill strategy                                               
BTCUSDT 4.0            optimal_k                       1.218637           99.7
XRPUSDT 13736.0        optimal_k                       3.625474          100.0
ETHUSDT 55.0           optimal_k                        2.66769           99.9
ADAUSDT 10436.0        optimal_k                       5.686933           98.4
SOLUSDT 315.0          optimal_k                       5.289195          100.0
...                                                         ...            ...
LTCUSDT 51.0           redistributed_second30_**5_80   5.280374           99.0
                       redistributed_second40_**5_80   5.296905           98.9
                       redistributed_second20_**10_80  5.440062           98.3
                       redistributed_second30_**10_80  5.440464           98.3
                       redistributed_second40_**10_80  5.451051           98.3

[532 rows x 2 columns]

In [19]:
benchmark.loc[data_assets[0]]

bps percent_filled
Volume to Fill strategy                                               
4.0            optimal_k                       1.218637           99.7
               capped_linear_mean              1.306596          100.0
               capped_**2_mean                 1.224362          100.0
               capped_**3_mean                 1.188136           99.8
               capped_**5_mean                 1.174523           98.6
...                                                 ...            ...
               redistributed_second30_**5_80   1.168541           99.2
               redistributed_second40_**5_80     1.1885           98.6
               redistributed_second20_**10_80  1.225911           95.8
               redistributed_second30_**10_80  1.225435           95.9
               redistributed_second40_**10_80  1.231775           95.5

[76 rows x 2 columns]

## Redistribute if Cap Exceeded but Decrease Caps based on Distance to Final Sec

maybe we could also include this volatility marker in there somehow. if volatility std is high, we make the caps decrease slower. if std is low, we decrease quickly. this should be unique for each id.

so the final model is:

**cap**(**function**) + **redistribute**(**decreasing caps** where **slope = 1 / volatility std**) -> 5 components

note that function could be replaced with Maaz's adaptive K approach

This is the ultimate model lmao

In [20]:
def vol_regime(x):
        low = x.quantile(0.2)
        high = x.quantile(0.8) 
        return pd.cut(
            x,
            bins=[-np.inf, low, high, np.inf],
            labels=["low", "normal", "high"]
        )

volatility_window = 120 # last 2 minutes volatility

for i in X_dfs:
    X_df = X_dfs[i]

    X_df["midprice"] = (X_df["ask_price_1"] + X_df["bid_price_1"]) / 2

    # volatility measure
    X_df["midvolatility"] = X_df.groupby("anonymized_id")["midprice"].transform(lambda x: np.log(x).diff()).fillna(0)

    X_df["midvolatilitystd"] = X_df.groupby("anonymized_id")["midvolatility"].transform(lambda x: x.rolling(volatility_window).std()) 

    X_df["vol_regime"] = X_df.groupby("anonymized_id")["midvolatilitystd"].transform(vol_regime)

In [21]:
last_sec = X_df["time_in_hour"].sort_values().iloc[-1]

In [22]:
benchmark = benchmark_copy.copy()

In [23]:
for data_asset in data_assets:

    X_df = X_dfs[data_asset]
    y_df = y_dfs[data_asset]
    ids = assets_ids[data_asset]
    volume_to_fill = volumes_to_fill[data_asset]

    for percentile in percentiles:
        for func in functions:

            bpss = []
            percents_filled = []

            for anon_id in ids:

                df_inst = y_df[y_df["anonymized_id"] == anon_id].sort_values("time_in_hour")
                X_df_inst = X_df[X_df["anonymized_id"] == anon_id].sort_values("time_in_hour")

                ###
                vol_regime = X_df[X_df["anonymized_id"] == anon_id][X_df["time_in_hour"] == last_sec]["vol_regime"].iloc[0]

                if vol_regime == "low":
                    volatility_cap = 45
                elif vol_regime == "high":
                    volatility_cap = 15
                elif vol_regime == "normal":
                    volatility_cap = 30
                ###

                if percentile == "mean":
                    volume_cap = np.mean(X_df_inst[cap_col])
                else:
                    volume_cap = np.percentile(X_df_inst[cap_col], percentile)

                if volume_cap == 0:
                    print("mean used!!!")
                    volume_cap = np.mean(X_df_inst[cap_col])
                
                volume_caps = np.append(np.array([0]*volatility_cap), np.linspace(0, volume_cap, 60-volatility_cap)) # linearly increasing caps from second_cap onwards

                ask_prices = df_inst[ASK_PRICE_COLS].to_numpy()
                ask_vols = df_inst[ASK_VOL_COLS].to_numpy()
                bid_prices = df_inst[BID_PRICE_COLS].to_numpy()
                bid_vols = df_inst[BID_VOL_COLS].to_numpy()
                close_price = df_inst['close'].dropna().iloc[-1]
                

                weights = np.linspace(0, 1, 60)

                if func != "linear":
                    weights = weights ** int(func[2:])

                id_positions = capped_allocation(weights, volume_to_fill, volume_caps)

                model_vol, model_avg_price = simulate_walk_the_book(
                    id_positions, ask_prices, ask_vols, bid_prices, bid_vols
                )
                
                if model_vol > 0 and not np.isnan(model_avg_price):
                    rel_error = np.abs(model_avg_price - close_price) / close_price
                    vol_penalty = min(100.0, volume_to_fill / model_vol)

                    bpss.append(rel_error * vol_penalty * 10000)
                    percents_filled.append(model_vol / volume_to_fill * 100)

            strategy = f"redistributed_volcap_{func}_{percentile}"
            benchmark.loc[(data_asset, volume_to_fill, strategy), "bps"] = np.mean(bpss)
            benchmark.loc[(data_asset, volume_to_fill, strategy), "percent_filled"] = np.round(np.mean(percents_filled), 1)

/var/folders/gh/k7rgm7qs3zl2lw8bn_ydlg_40000gq/T/ipykernel_2231/1728494756.py:67: PerformanceWarning: indexing past lexsort depth may impact performance.
  benchmark.loc[(data_asset, volume_to_fill, strategy), "bps"] = np.mean(bpss)
/var/folders/gh/k7rgm7qs3zl2lw8bn_ydlg_40000gq/T/ipykernel_2231/1728494756.py:68: PerformanceWarning: indexing past lexsort depth may impact performance.
  benchmark.loc[(data_asset, volume_to_fill, strategy), "percent_filled"] = np.round(np.mean(percents_filled), 1)
/var/folders/gh/k7rgm7qs3zl2lw8bn_ydlg_40000gq/T/ipykernel_2231/1728494756.py:67: PerformanceWarning: indexing past lexsort depth may impact performance.
  benchmark.loc[(data_asset, volume_to_fill, strategy), "bps"] = np.mean(bpss)
/var/folders/gh/k7rgm7qs3zl2lw8bn_ydlg_40000gq/T/ipykernel_2231/1728494756.py:68: PerformanceWarning: indexing past lexsort depth may impact performance.
  benchmark.loc[(data_asset, volume_to_fill, strategy), "percent_filled"] = np.round(np.mean(percents_filled), 1

# Ideal Strategy Analysis

## Best Approaches

In [24]:
def get_top_with_optimal(benchmark, asset, top_n=3):

    df = benchmark.xs(asset, level="Asset", drop_level=False)

    df_no_opt = df[df.index.get_level_values("strategy") != "optimal_k"]
    top = df_no_opt.sort_values("bps").head(top_n)

    optimal = df[df.index.get_level_values("strategy") == "optimal_k"]

    result = pd.concat([top, optimal])

    result = result.sort_values("bps")

    return result

In [25]:
get_top_with_optimal(benchmark, data_assets[0], top_n=1)

bps percent_filled
Asset   Volume to Fill strategy                                      
BTCUSDT 4.0            redistributed_**10_50  1.151176           99.2
                       optimal_k              1.218637           99.7

In [26]:
get_top_with_optimal(benchmark, data_assets[1], top_n=1)

bps percent_filled
Asset   Volume to Fill strategy                                            
XRPUSDT 13736.0        redistributed_volcap_**5_80  3.558497           99.8
                       optimal_k                    3.625474          100.0

In [27]:
get_top_with_optimal(benchmark, data_assets[2], top_n=1)

bps percent_filled
Asset   Volume to Fill strategy                                     
ETHUSDT 55.0           redistributed_**3_80  2.583682           99.8
                       optimal_k              2.66769           99.9

In [28]:
get_top_with_optimal(benchmark, data_assets[3], top_n=1)

bps percent_filled
Asset   Volume to Fill strategy                                            
ADAUSDT 10436.0        redistributed_volcap_**5_80   5.52503           99.0
                       optimal_k                    5.686933           98.4

In [29]:
get_top_with_optimal(benchmark, data_assets[4], top_n=1)

bps percent_filled
Asset   Volume to Fill strategy                                            
SOLUSDT 315.0          redistributed_volcap_**5_80  5.268796           99.9
                       optimal_k                    5.289195          100.0

In [30]:
get_top_with_optimal(benchmark, data_assets[5], top_n=1)

bps percent_filled
Asset    Volume to Fill strategy                                     
DOGEUSDT 60349.0        redistributed_**3_80  4.908375           99.9
                        optimal_k             5.000712           99.9

In [31]:
get_top_with_optimal(benchmark, data_assets[6], top_n=1)

bps percent_filled
Asset   Volume to Fill strategy                                               
LTCUSDT 51.0           redistributed_volcap_linear_50  5.202131           99.5
                       optimal_k                       5.302726           99.4